# Comparación odometría vs EKF vs ground truth

Todas las poses se llevan al frame `map` mediante composición de transforms:

| Pose | Cadena |
|---|---|
| Odometría | `map → odom → base_link` |
| EKF | `map → base_link_ekf` |
| Ground truth | `map → odom → base_link_gt` |

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rosbags.rosbag2 import Reader
from rosbags.typesys import Stores, get_typestore

BAG_PATH = Path("exp_odom_ekf")

In [ ]:
typestore = get_typestore(Stores.ROS2_HUMBLE)

# Almacenamos cada transform por separado: {child_frame: [(t_ns, x, y, yaw)]}
from collections import defaultdict
import math

def quat_to_yaw(q) -> float:
    """Extrae yaw de un quaternion."""
    siny = 2.0 * (q.w * q.z + q.x * q.y)
    cosy = 1.0 - 2.0 * (q.y * q.y + q.z * q.z)
    return math.atan2(siny, cosy)

# raw transforms: child_frame_id -> list of (t_ns, parent, x, y, yaw)
raw = defaultdict(list)

with Reader(BAG_PATH) as reader:
    for connection, t_ns, rawdata in reader.messages():
        msg = typestore.deserialize_cdr(rawdata, connection.msgtype)
        for tf in msg.transforms:
            t = tf.transform.translation
            r = tf.transform.rotation
            raw[tf.child_frame_id].append((
                t_ns,
                tf.header.frame_id,
                t.x, t.y,
                quat_to_yaw(r)
            ))

def to_df(child: str) -> pd.DataFrame:
    rows = raw[child]
    df = pd.DataFrame(rows, columns=["t_ns", "parent", "x", "y", "yaw"])
    return df.sort_values("t_ns").reset_index(drop=True)

tf_map_odom       = to_df("odom")          # map → odom
tf_odom_base      = to_df("base_link")     # odom → base_link
tf_odom_gt        = to_df("base_link_gt")  # odom → base_link_gt
tf_map_ekf        = to_df("base_link_ekf") # map  → base_link_ekf

print("map→odom:       ", len(tf_map_odom))
print("odom→base_link: ", len(tf_odom_base))
print("odom→base_link_gt:", len(tf_odom_gt))
print("map→base_link_ekf:", len(tf_map_ekf))

In [ ]:
def compose_2d(x1, y1, yaw1, x2, y2, yaw2):
    """Composición de transforms 2D: T1 * T2."""
    c, s = np.cos(yaw1), np.sin(yaw1)
    x = x1 + c * x2 - s * y2
    y = y1 + s * x2 + c * y2
    yaw = yaw1 + yaw2
    return x, y, yaw

def interpolate(df: pd.DataFrame, t_ns: np.ndarray) -> pd.DataFrame:
    """Interpola x, y, yaw de df en los timestamps t_ns (ZOH)."""
    idx = np.searchsorted(df.t_ns.values, t_ns, side="right") - 1
    idx = np.clip(idx, 0, len(df) - 1)
    return df.iloc[idx][["x", "y", "yaw"]].values

# Usamos los timestamps de base_link_gt como referencia
t_ref = tf_odom_gt.t_ns.values
t0 = t_ref[0]
t_s = (t_ref - t0) * 1e-9

# map → odom interpolado en t_ref
map_odom = interpolate(tf_map_odom, t_ref)        # (N,3)

# odom → base_link interpolado en t_ref
odom_base = interpolate(tf_odom_base, t_ref)

# odom → base_link_gt en t_ref (sin interpolación, es la referencia)
gt_xy = tf_odom_gt[["x", "y", "yaw"]].values

# map → base_link_ekf interpolado en t_ref
ekf_xy = interpolate(tf_map_ekf, t_ref)

# Componer: map → base_link = (map→odom) * (odom→base_link)
odom_in_map = np.array([compose_2d(*map_odom[i], *odom_base[i]) for i in range(len(t_ref))])

# Componer: map → base_link_gt = (map→odom) * (odom→base_link_gt)
gt_in_map   = np.array([compose_2d(*map_odom[i], *gt_xy[i])   for i in range(len(t_ref))])

# EKF ya está en map
ekf_in_map = ekf_xy

print(f"Puntos de referencia: {len(t_ref)}, duración: {t_s[-1]:.1f} s")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

ax.plot(gt_in_map[:, 0],   gt_in_map[:, 1],   label="ground truth",  linewidth=2,   color="k")
ax.plot(odom_in_map[:, 0], odom_in_map[:, 1], label="odometría",      linewidth=1.5, color="tab:blue",   linestyle="--")
ax.plot(ekf_in_map[:, 0],  ekf_in_map[:, 1],  label="EKF",            linewidth=1.5, color="tab:orange", linestyle="--")

# Marcar inicio
ax.plot(*gt_in_map[0, :2],  "ko", markersize=8)

ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_aspect("equal")
ax.legend()
ax.grid(True, linewidth=0.4)
ax.set_title("Trayectoria en frame map")
fig.tight_layout()
plt.savefig("../docs/assets/trayectoria_comparacion.png", dpi=150)
plt.show()

In [ ]:
e_odom = np.linalg.norm(odom_in_map[:, :2] - gt_in_map[:, :2], axis=1)
e_ekf  = np.linalg.norm(ekf_in_map[:, :2]  - gt_in_map[:, :2], axis=1)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t_s, e_odom, label="error odometría",  linewidth=1.5, color="tab:blue")
ax.plot(t_s, e_ekf,  label="error EKF",         linewidth=1.5, color="tab:orange")
ax.set_xlabel("tiempo [s]")
ax.set_ylabel("error de posición [m]")
ax.legend()
ax.grid(True, linewidth=0.4)
ax.set_title("Error de posición respecto al ground truth")
fig.tight_layout()
plt.savefig("../docs/assets/error_posicion.png", dpi=150)
plt.show()

print(f"RMSE odometría: {np.sqrt((e_odom**2).mean()):.4f} m")
print(f"RMSE EKF:       {np.sqrt((e_ekf**2).mean()):.4f} m")
print(f"Error final odometría: {e_odom[-1]:.4f} m")
print(f"Error final EKF:       {e_ekf[-1]:.4f} m")